### Tool calling
- connection between tool and LLM
- Tool binding 
- LLM knows all the available tools
- LLM know what input to expect

In [2]:
# Model calling and intial setup
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser 
import warnings
warnings.filterwarnings("ignore") 

load_dotenv()
# Load env
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
AZURE_BASE_URL = os.getenv("AZURE_BASE_URL")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_CHAT_DEPLIOYMENT_NAME = os.getenv("AZURE_CHAT_DEPLIOYMENT_NAME")

parser = StrOutputParser()

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)

llm_openai = AzureChatOpenAI(
    model="gpt-4o-mini",                         
    deployment_name=AZURE_CHAT_DEPLIOYMENT_NAME ,  # deployment name in Azure
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_BASE_URL,
    api_version="2024-02-01"
    )
llm_openai.invoke("What are your creater, also what type of LLM are you").content
# llm_gemini.invoke("who is father of india").content

'I was created by OpenAI, an artificial intelligence research organization. I am based on the GPT-3.5 architecture, which is a type of large language model (LLM). My main function is to understand and generate human-like text based on the input I receive. If you have any specific questions or need assistance, feel free to ask!'

In [12]:
from langchain_core.tools import tool

# Creating a tool
@tool 
def multiply(a : int , b :int)->int:
    """fucntion to add two numbers"""
    return a * b

results = multiply.invoke({"a" : 3 , "b":4})

multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [17]:
# Now tool binding
llm_with_multiply_tool =  llm_gemini.bind_tools([multiply])

#### Tool Calling

In [ ]:
# when LLM need to call a tool with 
results = llm_with_multiply_tool.invoke("Can you multliply 8 with 10 ")

In [25]:
results.tool_calls[0] 

{'name': 'multiply',
 'args': {'a': 8.0, 'b': 10.0},
 'id': 'd3a606e2-a6ca-41e3-ba55-768e591b9e63',
 'type': 'tool_call'}

#### Tool Execution
- LLM doesn't run the tool, it only give the schema to run a tool
- tool execution is on the person using the tool

In [26]:
# Actual python fucntion, that run using args suggested by the LLM
multiply.invoke(results.tool_calls[0].get("args"))

80

In [ ]:
# Tool message
multiply.invoke(results.tool_calls[0] )

# In output we get the ToolMessage that we can again put in the LLM

ToolMessage(content='80', name='multiply', tool_call_id='d3a606e2-a6ca-41e3-ba55-768e591b9e63')